# HiveMind Benchmark auf Colab A100 — v2

Runtime: **A100 GPU** (40 GB reichen für Qwen3.6-35B Q4), High-RAM, ≥100 GB Disk.

Das ist der **echte HiveMind-Harness**:
- klont den aktuellen `dev`-Branch,
- installiert Python 3.14, HiveMind, llama.cpp **CUDA 12.8** (Treiber-kompatibel), Qwen3.6-35B, Sharp-Template, Aider,
- startet die Engine, lädt das Modell vor und erkennt den llama-Port automatisch,
- Smoke: 3 Tasks × HiveMind/raw/Aider,
- optional 21 Tasks und 5 DeepSWE-Tasks mit HiveMind + Modal-Verifier.

**Kein Key** für den 21-Task-Lauf. Nur der optionale DeepSWE-Teil braucht einen Modal-Token (Zelle „Modal token“).


In [ ]:
# 1) GPU + Disk
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1


In [ ]:
# 2) Clone/Update the pushed dev branch + bootstrap
![ -d /root/HiveMind_llama/.git ] || git clone --branch dev https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git pull --ff-only
!cp /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='%h %s'


In [ ]:
# 3) Setup (~15-30 min: Python 3.14, llama.cpp CUDA 12.8, 20 GB model, aider)
# CUDA 12.8 is deliberately pinned: the A100 driver may report 13.0, while the
# newest nightly can be a 13.5 binary that does not start.
!HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup


In [ ]:
# 4) Bootstrap-Version prüfen, dann Engine starten und 35B laden
!grep -q 'vram_budget_gb' /root/rent_bootstrap.sh && echo BOOTSTRAP_OK || echo BOOTSTRAP_STALE
!bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health


## Falls der llama-server nicht startet

Prüfe zuerst das neueste Server-Log:

```bash
ls -t /root/HiveMind_llama/logs/llama_server_*.log | head -1
tail -80 "$(ls -t /root/HiveMind_llama/logs/llama_server_*.log | head -1)"
```

Bei CUDA-/Treiberfehlern hilft dieser Fallback:

```python
!cd /root/HiveMind_llama && .venv/bin/python deploy/fetch_llamacpp.py --backend cuda --cuda-version 12.8 --force
!bash /root/rent_bootstrap.sh stop
!bash /root/rent_bootstrap.sh serve
```


In [ ]:
# 5) Smoke: 3 tasks x 3 conditions (HiveMind, raw, Aider)
# The runner is now unbuffered: output appears per cell.
!bash /root/rent_bootstrap.sh smoke


In [ ]:
# 6) Full 21-task run in tmux
import os
os.environ['HM_RUNS'] = '1'
os.environ['HM_CONDITIONS'] = 'hivemind_duo,raw,aider'
os.environ['HM_TASKS'] = 'bench/tasks'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 7) Progress (re-run as often as you like)
!bash /root/rent_bootstrap.sh status


## DeepSWE-5 mit HiveMind (optional)

HiveMind läuft über `/stream`; nur der DeepSWE-Verifier läuft auf Modal. Dafür brauchst du einen Modal-Token. Für den 21-Task-Lauf kannst du diese Zellen überspringen.


In [ ]:
# 8) Modal token — only for DeepSWE
MODAL_TOKEN_ID = ""       # <<< HIER Token ID
MODAL_TOKEN_SECRET = ""   # <<< HIER Token Secret

import os
MODAL_READY = bool(MODAL_TOKEN_ID and MODAL_TOKEN_SECRET)
if MODAL_READY:
    os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
    os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET
    print("Modal token present:", MODAL_TOKEN_ID[:8] + "...")
else:
    print("No Modal token: DeepSWE cells will be skipped.")


In [ ]:
# 9) DeepSWE tasks + Modal SDK in the HiveMind venv
import os, subprocess
if not MODAL_READY:
    print("SKIP: no Modal token")
else:
    if not os.path.isdir("/root/deep-swe/.git"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/datacurve-ai/deep-swe.git", "/root/deep-swe"], check=True)
    subprocess.run("cd /root/HiveMind_llama && (/root/.local/bin/uv pip install --python .venv/bin/python -q modal || uv pip install --python .venv/bin/python -q modal)", shell=True, check=True)


In [ ]:
# 10) HiveMind on 5 DeepSWE tasks, verifier on Modal
import subprocess
if not MODAL_READY:
    print("SKIP: no Modal token")
else:
    subprocess.run(
        "cd /root/HiveMind_llama && .venv/bin/python bench/external/deepswe_hivemind.py "
        "--tasks /root/deep-swe/tasks --hivemind http://127.0.0.1:8001 "
        "--model qwen3.6:35b-a3b-ud --n-tasks 5 --sample-seed 0 "
        "--verifier modal --out /root/HiveMind_llama/bench/results/deepswe-hivemind-5",
        shell=True, check=True)


In [ ]:
# 11) Save results to Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hivemind_bench
!cp -v /root/HiveMind_llama/bench/results/*.json /content/drive/MyDrive/hivemind_bench/
!cp -v /root/HiveMind_llama/bench_full.log /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true


In [ ]:
# 12) Stop engine + model; then terminate the runtime/rental.
!bash /root/rent_bootstrap.sh stop
